# SciLite Ground Truth Generator

Fetches **Europe PMC SciLite annotations** for a set of PMC articles and builds a ground-truth dataset that can be used to evaluate the data-gatherer pipeline.

SciLite annotations cover entity types such as:
- **Accession Numbers** — dataset/sequence accessions (GEO, UniProt, PDB, …)
- **Gene / Protein** mentions
- **Disease**, **Chemical**, **Organism** mentions
- **Gene Ontology** terms

The accession-number annotations are the primary ground truth for the dataset-citation extraction task.

**API limit:** 8 article IDs per request.  
**Script:** `scripts/scrape_annotations.py` (all fetch logic lives there — this notebook just orchestrates and explores).

In [ ]:
import sys, json, logging
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path().resolve()))  # project root

from scripts.scrape_annotations import (
    load_ids_from_csv,
    make_session,
    fetch_annotations,
    load_checkpoint,
    extract_accession_numbers,
    to_dataframe,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

## Configuration

In [ ]:
# --- Input ---
PMC_ID_FILE   = "k8s/input/article_ids_REV_pmc.csv"   # 118 k articles
EVAL_ID_FILE  = "k8s/input/article_ids_eval.csv"       # smaller eval set

# --- Output ---
OUTPUT_DIR    = Path("scripts/output")
OUTPUT_SUBSET = OUTPUT_DIR / "annotations_subset.json"
OUTPUT_FULL   = OUTPUT_DIR / "annotations_full.json"
OUTPUT_GT_CSV = OUTPUT_DIR / "scilite_ground_truth.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Fetch params ---
BATCH_SIZE = 8    # hard API limit
PAUSE      = 0  # seconds between requests

## 1. Load PMC IDs

In [ ]:
pmc_ids = load_ids_from_csv(PMC_ID_FILE)
print(f"Loaded {len(pmc_ids):,} PMC IDs from {PMC_ID_FILE}")
print("Sample:", pmc_ids[:5])

## 2. Fetch Annotations

### 2a. Subset run (quick test)

In [ ]:
SUBSET_N = 16
subset   = pmc_ids[:SUBSET_N]

session = make_session()
done = fetch_annotations(subset, session, batch_size=BATCH_SIZE, pause=PAUSE, output=OUTPUT_SUBSET)

with OUTPUT_SUBSET.open("w") as fh:
    json.dump(done, fh, indent=2)

print(f"\nFetched {len(done)} articles → {OUTPUT_SUBSET}")
for pmcid, anns in done.items():
    print(f"  {pmcid}: {len(anns):>4d} annotations")

### 2b. Full run (118 k articles, resumable)

Estimated time at batch=8 / pause=0.2 s: **~15 hours**.  
Interrupt and re-run freely — the checkpoint file resumes automatically.

In [ ]:
done_full = load_checkpoint(OUTPUT_FULL)  # resumes from prior run if present
print(f"Checkpoint: {len(done_full):,} articles already fetched")

session = make_session()
done_full = fetch_annotations(
    pmc_ids, session,
    batch_size=BATCH_SIZE, pause=PAUSE,
    done=done_full, output=OUTPUT_FULL,
)

with OUTPUT_FULL.open("w") as fh:
    json.dump(done_full, fh, indent=2)
print(f"Done. {len(done_full):,} articles → {OUTPUT_FULL}")

## 3. Explore Results

Run on the subset output for a quick sanity-check; swap `done` → `done_full` for the full dataset.

In [ ]:
import json, pandas as pd
from pathlib import Path
from scripts.scrape_annotations import extract_accession_numbers, to_dataframe

with open("scripts/output/annotations_accession.json") as fh:
    data = json.load(fh)

acc = extract_accession_numbers(data)
gt  = to_dataframe(acc)

print(f"Articles with accessions: {sum(1 for v in acc.values() if v):,} / {len(acc):,}")
print(f"Total rows: {len(gt):,}")

gt.to_parquet("scripts/output/scilite_ground_truth.parquet", index=False)
print("Saved → scripts/output/scilite_ground_truth.parquet")

In [ ]:
# # explore subtype distribution
# print("\nSubtype distribution:")

# print(gt["subType"].value_counts())

In [ ]:
total_articles    = len(data)
articles_with_ann = sum(1 for v in data.values() if v)
total_annotations = sum(len(v) for v in data.values())

print(f"Articles:             {total_articles:>6,}")
print(f"  with annotations:   {articles_with_ann:>6,}  ({articles_with_ann/total_articles:.0%})")
print(f"Total annotations:    {total_annotations:>6,}")
print(f"Avg per article:      {total_annotations/max(articles_with_ann,1):>6.1f}")

In [ ]:
# Show a few raw annotations from the first article that has some
for pmcid, anns in data.items():
    if anns:
        print(f"{pmcid}  ({len(anns)} annotations — showing first 5)")
        for ann in anns[:5]:
            print(f"  exact={ann.get('exact')!r:30s}  section={ann.get('section','?')!r}")
            for tag in ann.get("tags", []):
                print(f"    → {tag.get('name')!r}  uri={tag.get('uri','')[:60]}")
        break

## 5. Evaluate K8S Iterations Against SciLite Ground Truth

In [ ]:
import sys, time
from pathlib import Path
import pandas as pd
from scripts.experiment_utils import *
from data_gatherer.data_gatherer import DataGatherer

# Subtypes that are ontology annotations, NOT deposited dataset accessions.
# Excluding them gives a realistic recall ceiling for what our system can find.
NON_DATA_SUBTYPES = {
    "Gene Ontology (GO)",  # 77k  — functional annotations, not dataset deposits
    "RefSNP",              # 27k  — variant IDs (dbSNP), not dataset accessions
    "Pfam",                # 2.5k — protein family annotations
    "InterPro",            # 1.9k — protein domain annotations
    "HGNC",                # 687  — gene name identifiers
    "Brenda",              # 447  — enzyme database annotations
    "Rfam",                # 306  — RNA family annotations
    "EFO",                 # 61   — Experimental Factor Ontology terms
    "Treefam",             # 6    — phylogenetic family annotations
}

gt_raw = pd.read_parquet("scripts/output/scilite_ground_truth.parquet")
n_raw = len(gt_raw)
gt = gt_raw[~gt_raw["subType"].isin(NON_DATA_SUBTYPES)].copy()
gt = gt.rename(columns={"exact": "identifier"})

n_removed = n_raw - len(gt)
print(f"SciLite GT (raw):      {n_raw:,} rows  |  {gt_raw['pmcid'].nunique():,} articles")
print(f"SciLite GT (filtered): {len(gt):,} rows  |  {gt['pmcid'].nunique():,} articles")
print(f"Removed {n_removed:,} non-data annotations ({n_removed/n_raw:.1%})")
print("\nRemaining subType distribution:")
print(gt["subType"].value_counts().to_string())

dg = DataGatherer(
    llm_name="hf-vida-nyu/flan-t5-base-dataref-info-extract",
    save_to_cache=False,
    load_from_cache=False,
    log_level="WARNING",
)

In [ ]:
import time

results = []

for iter_dir in sorted(Path("k8s/output").glob("iter*"), key=lambda p: int(p.name[4:])):
    csv_path = iter_dir / "dataset_citations.csv"
    print(f"\nEvaluating {iter_dir.name}...")
    if not csv_path.exists():
        print(f"  [skip] {iter_dir.name}: no dataset_citations.csv")
        continue

    ret_df = pd.read_csv(csv_path)
    if ret_df.empty:
        print(f"  [skip] {iter_dir.name}: empty")
        continue

    iteration = int(iter_dir.name[4:])
    fp_file = f"scripts/output/scilite_fp_iter{iteration}.txt"
    fn_file = f"scripts/output/scilite_fn_iter{iteration}.txt"

    t0 = time.time()
    metrics = evaluate_performance(
        ret_df, gt, dg,
        fp_file,
        false_negatives_file=fn_file,
        repo_return=True,
        gt_base=list(set(ret_df["source_url"].tolist())),
    )
    elapsed = time.time() - t0
    metrics["iteration"] = iteration
    metrics["n_articles"] = ret_df["source_url"].nunique()
    results.append(metrics)
    print(f"  iter{iteration:>2d}  P={metrics['average_precision']:.3f}  "
          f"R={metrics['average_recall']:.3f}  F1={metrics['f1_score']:.3f}  "
          f"({metrics['n_articles']:,} articles, {elapsed:.1f}s)")

results_df = pd.DataFrame(results).sort_values("iteration").reset_index(drop=True)
print(f"\n{results_df.to_string(index=False)}")

## 6. Evaluate Against Gold Repository GT

Second evaluation pass using curated repository records as ground truth:
- **GEO** — dataset IDs from GEO API (352 k records)
- **PRIDE / ProteomeXchange** — from ProteomeXchange search (44 k records)
- **Synapse** — from Europe PMC Synapse ID mining (4 k records)

Coverage: ~120 k unique PMC articles. Uses the same `evaluate_performance()` as Section 5.

In [ ]:
import pandas as pd
from pathlib import Path

gold_gt = pd.read_parquet("scripts/output/gold/dataset_citation_records_Table.parquet")
gold_gt = gold_gt[gold_gt['pmcid'].notna() & (gold_gt['pmcid'].str.strip() != '')].copy()
gold_gt['identifier'] = gold_gt['identifier'].str.strip()

print(f"Gold GT: {len(gold_gt):,} rows  |  {gold_gt['pmcid'].nunique():,} articles")
print("Repositories:")
for repo, n in gold_gt['repository'].value_counts().items():
    print(f"  {repo:<20s} {n:>7,}")
print()
print(gold_gt.head(3).to_string(index=False))

In [ ]:
import time

results_gold = []

for iter_dir in sorted(Path("k8s/output").glob("iter*"), key=lambda p: int(p.name[4:])):
    csv_path = iter_dir / "dataset_citations.csv"
    print(f"\nEvaluating {iter_dir.name} (gold GT)...")
    if not csv_path.exists():
        print(f"  [skip] {iter_dir.name}: no dataset_citations.csv")
        continue

    ret_df = pd.read_csv(csv_path)
    if ret_df.empty:
        print(f"  [skip] {iter_dir.name}: empty")
        continue

    iteration = int(iter_dir.name[4:])
    fp_file  = f"scripts/output/gold_fp_iter{iteration}.txt"
    fn_file  = f"scripts/output/gold_fn_iter{iteration}.txt"

    t0 = time.time()
    metrics = evaluate_performance(
        ret_df, gold_gt, dg,
        fp_file,
        false_negatives_file=fn_file,
        repo_return=True,
        gt_base=list(set(ret_df["source_url"].tolist())),
    )
    elapsed = time.time() - t0
    metrics["iteration"] = iteration
    metrics["n_articles"] = ret_df["source_url"].nunique()
    results_gold.append(metrics)
    print(f"  iter{iteration:>2d}  P={metrics['average_precision']:.3f}  "
          f"R={metrics['average_recall']:.3f}  F1={metrics['f1_score']:.3f}  "
          f"({metrics['n_articles']:,} articles, {elapsed:.1f}s)")

results_gold_df = pd.DataFrame(results_gold).sort_values("iteration").reset_index(drop=True)
print(f"\n{results_gold_df.to_string(index=False)}")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

METRIC_STYLES = [
    ("average_precision", "#2196F3", "Precision"),
    ("average_recall",    "#4CAF50", "Recall"),
    ("f1_score",          "#FF9800", "F1"),
]

# --- Side-by-side: SciLite vs Gold ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["SciLite GT (Europe PMC annotations)", "Gold Repository GT (GEO / PRIDE / Synapse)"],
    shared_yaxes=True,
)
for col, (df, label) in enumerate([(results_df, "SciLite"), (results_gold_df, "Gold")], start=1):
    for metric, color, name in METRIC_STYLES:
        fig.add_trace(go.Scatter(
            x=df["iteration"], y=df[metric],
            mode="lines+markers", name=name,
            line=dict(color=color, width=2), marker=dict(size=7),
            showlegend=(col == 1),
            hovertemplate=f"{name}: %{{y:.3f}}<br>iter %{{x}}<extra></extra>",
        ), row=1, col=col)

fig.update_yaxes(range=[0, 1])
fig.update_xaxes(dtick=1)
fig.update_layout(
    title="Data-gatherer performance per iteration — two GT sources",
    legend_title="Metric", width=1300, height=500,
)
fig.show()
fig.write_html("scripts/output/scilite_vs_gold_eval.html")
print("Saved → scripts/output/scilite_vs_gold_eval.html")

# --- Overlay F1 only ---
fig2 = go.Figure()
for df, label, dash in [(results_df, "SciLite", "solid"), (results_gold_df, "Gold", "dash")]:
    fig2.add_trace(go.Scatter(
        x=df["iteration"], y=df["f1_score"],
        mode="lines+markers", name=f"F1 ({label})",
        line=dict(width=2, dash=dash), marker=dict(size=7),
        hovertemplate=f"F1 ({label}): %{{y:.3f}}<br>iter %{{x}}<extra></extra>",
    ))
fig2.update_layout(
    title="F1 score comparison: SciLite vs Gold GT",
    xaxis_title="Iteration", yaxis_title="F1",
    yaxis=dict(range=[0, 1]), xaxis=dict(dtick=1),
    width=900, height=450,
)
fig2.show()
fig2.write_html("scripts/output/f1_comparison.html")
print("Saved → scripts/output/f1_comparison.html")

In [ ]:
#  Analyze what we are missing --> 